In [1]:
!pip install argon2-cffi -q

In [7]:
import os
import json
from google.colab import drive
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError, HashingError

class Sistema:
    def __init__(self, drive_path="/content/drive/MyDrive/Segurança da Informação/Entrega 2"):
        drive.mount('/content/drive', force_remount=False)

        self.db_dir = drive_path
        self.db_file = os.path.join(self.db_dir, "usuarios_xpresso.json")
        os.makedirs(self.db_dir, exist_ok=True)

        # Instancia o Argon2id com parâmetros seguros recomendados pelo OWASP
        self.ph = PasswordHasher(
            time_cost=2,        # Número de iterações
            memory_cost=65536,  # Uso de memória em KiB (64 MB)
            parallelism=1       # Threads paralelas
        )
        self.banco_de_dados = self._carregar_banco()

    def _carregar_banco(self):
        if os.path.exists(self.db_file):
            try:
                with open(self.db_file, 'r', encoding='utf-8') as f:
                    print(f"json acessado")
                    return json.load(f)
            except Exception as e:
                print(f"erro no drive {e}")
                return {}
        print("Novo json criado")
        return {}

    def _salvar_banco(self):
        try:
            with open(self.db_file, 'w', encoding='utf-8') as f:
                json.dump(self.banco_de_dados, f, indent=4, ensure_ascii=False)
            print(f"dados salvos no arquivo: {self.db_file}")
        except Exception as e:
            print(f"problema no google drive {e}")

    def registrar_jogador(self, usuario: str, senha: str):
        if usuario in self.banco_de_dados:
            print(f"\n[Xpresso MMORPG] Erro: O personagem/conta '{usuario}' já está cadastrado!")
            return False

        try:
            # hash
            hash_senha = self.ph.hash(senha)
            self.banco_de_dados[usuario] = hash_senha

            # save
            self._salvar_banco()
            print(f"Conta do jogador '{usuario}' criado com sucesso no servidor")
            return True
        except HashingError as e:
            print(f"Falha na biblioteca de hash")
            return False

    def logar_jogador(self, usuario: str, senha_tentativa: str):
        if usuario not in self.banco_de_dados:
            print(f"Falha no Login, conta '{usuario}' não encontrada.")
            return False
        hash_armazenado = self.banco_de_dados[usuario]
        try:
            if self.ph.verify(hash_armazenado, senha_tentativa):
                print(f"\nLogin realizado, bem-vindo de volta ao xpresso, {usuario}!")

                # (rehash)
                if self.ph.check_needs_rehash(hash_armazenado):
                    self.banco_de_dados[usuario] = self.ph.hash(senha_tentativa)
                    self._salvar_banco()
                return True
        except VerifyMismatchError:
            print(f"Falha no Login, senha incorreta para o usuário '{usuario}'.")
            return False
        except Exception as e:
            print(f"\n erro no codigo: {e}")
            return False

In [9]:
if __name__ == "__main__":
    servidor = Sistema()

    while True:
        print("     XPRESSO MMORPG - Login  ")
        print("[1] Registrar nova conta de aventureiro")
        print("[2] Fazer login no servidor")
        print("[3] Sair do jogo")

        opcao = input("\nEscolha sua ação (1, 2 ou 3): ").strip()

        if opcao == "1":
            print("\nREGISTRO DE NOVO PERSONAGEM")
            usuario = input("Digite o nome de usuário (ID): ").strip()
            senha = input("Digite a senha: ")

            if usuario and senha:
                servidor.registrar_jogador(usuario, senha)
            else:
                print("[Erro] O nome de usuário e a senha não podem estar em branco.")

        elif opcao == "2":
            print("\n--- LOGIN NO XPRESSO MMORPG ---")
            usuario = input("Digite o nome de usuário (ID): ").strip()
            senha = input("Digite a senha: ")

            if usuario and senha:
                servidor.logar_jogador(usuario, senha)
            else:
                print("[Erro] Preencha todos os campos.")

        elif opcao == "3":
            print("\nEncerrando jogo.")
            break
        else:
            print("\nEscolha entre 1, 2 ou 3.")

Conectando ao Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
json acessado
     XPRESSO MMORPG - Login  
[1] Registrar nova conta de aventureiro
[2] Fazer login no servidor
[3] Sair do jogo

Escolha sua ação (1, 2 ou 3): 3

Encerrando jogo.
